In [3]:
# [CELL 1] LOG ANALYSIS STUDENT CONFIGURATION
from pathlib import Path

LAST_NAME = "Dela Cruz"
STUDENT_ID = "TUPM-1251-26"
SEED_DIGIT = 6
ID_SUM = 17

RECORDS_TO_PROCESS = 1000 + ID_SUM  # 1017
TARGET_CLASS_PREFIX = "4"  # (6 % 4) + 2 = 4 (4xx Client Error class)

print("Records to Process:", RECORDS_TO_PROCESS)
print("Target Status Class Prefix:", TARGET_CLASS_PREFIX)


Records to Process: 1017
Target Status Class Prefix: 4


In [4]:
# [CELL 2] INITIALIZE ANALYSIS WORKSPACE AND PATHS
DATA_DIR = Path("data")
OUTPUT_DIR = Path("output")
BACKUP_DIR = Path("backup")

log_dataset_path = DATA_DIR / "weblog.csv"
filtered_csv_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_log_filtered.csv"
json_summary_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_log_summary.json"
json_backup_path = BACKUP_DIR / f"{LAST_NAME}_{STUDENT_ID}_log_summary.json"

print("Dataset path:", log_dataset_path)
print("Filtered output path:", filtered_csv_path)
print("JSON summary path:", json_summary_path)
print("Backup verification path:", json_backup_path)


Dataset path: data\weblog.csv
Filtered output path: output\Dela Cruz_TUPM-1251-26_log_filtered.csv
JSON summary path: output\Dela Cruz_TUPM-1251-26_log_summary.json
Backup verification path: backup\Dela Cruz_TUPM-1251-26_log_summary.json


In [5]:
# [CELL 3] READ LOG DATASET LINE BY LINE
processed_count = 0
target_class_records = 0
status_counts = {}
url_counts = {}
selected_rows = []

with open(log_dataset_path, "r", encoding="utf-8") as file:
    header = file.readline().strip().split(",")
    
    ip_idx = header.index("IP")
    time_idx = header.index("Time")
    url_idx = header.index("URL")
    status_idx = header.index("Staus")  # Matches column name typo in dataset
    
    for line in file:
        if processed_count >= RECORDS_TO_PROCESS:
            break
            
        fields = line.strip().split(",")
        processed_count += 1
        
        ip = fields[ip_idx]
        timestamp = fields[time_idx]
        url = fields[url_idx]
        status = fields[status_idx]
        
        if status.startswith(TARGET_CLASS_PREFIX):
            target_class_records += 1
            
            if status in status_counts:
                status_counts[status] = status_counts[status] + 1
            else:
                status_counts[status] = 1
                
            if url in url_counts:
                url_counts[url] = url_counts[url] + 1
            else:
                url_counts[url] = 1
                
            selected_rows.append([ip, timestamp, url, status])

print("Records processed from dataset:", processed_count)
print("Records matching target class:", target_class_records)


Records processed from dataset: 1017
Records matching target class: 5


In [6]:
# [CELL 4] EXTRACT FIVE MOST REQUESTED URLS
top_urls = {}
temp_urls = url_counts.copy()

for _ in range(5):
    if not temp_urls:
        break
    highest_url = None
    highest_count = -1
    for url in temp_urls:
        if temp_urls[url] > highest_count:
            highest_count = temp_urls[url]
            highest_url = url
    if highest_url is not None:
        top_urls[highest_url] = highest_count
        del temp_urls[highest_url]

print("Top 5 Requested URLs calculated:")
for url, count in top_urls.items():
    print(f"{url}: {count}")


Top 5 Requested URLs calculated:
GET /robots.txt HTTP/1.1: 4
GET /favicon.ico HTTP/1.1: 1


In [7]:
# [CELL 5] EXPORT FILTERED CSV AND SUMMARY JSON
import json

with open(filtered_csv_path, "w", encoding="utf-8") as out_csv:
    out_csv.write("IP,Time,URL,Status\n")
    for row in selected_rows:
        out_csv.write(f"{row[0]},{row[1]},{row[2]},{row[3]}\n")

summary_data = {
    "student_last_name": LAST_NAME,
    "student_id": STUDENT_ID,
    "records_processed": processed_count,
    "target_status_class": "4xx",
    "records_in_target_class": target_class_records,
    "status_code_counts": status_counts,
    "top_5_requested_urls": top_urls
}

with open(json_summary_path, "w", encoding="utf-8") as out_json:
    json.dump(summary_data, out_json, indent=4)

print("Filtered CSV created:", filtered_csv_path.exists())
print("JSON Summary file created:", json_summary_path.exists())


Filtered CSV created: True
JSON Summary file created: True


In [8]:
# [CELL 6] CREATE BACKUP AND RUN METADATA VERIFICATION
import shutil

shutil.copy2(json_summary_path, json_backup_path)

print("--- Final Assessment Results ---")
print("Records processed:", processed_count)
print("Target status class:", "4xx")
print("Records in target class:", target_class_records)
print("Count per status code:", status_counts)
print("Filtered CSV file:", filtered_csv_path.name)
print("JSON summary file:", json_summary_path.name)
print("Backup verification (File exists):", json_backup_path.exists())


--- Final Assessment Results ---
Records processed: 1017
Target status class: 4xx
Records in target class: 5
Count per status code: {'404': 5}
Filtered CSV file: Dela Cruz_TUPM-1251-26_log_filtered.csv
JSON summary file: Dela Cruz_TUPM-1251-26_log_summary.json
Backup verification (File exists): True
